# Demo guiada — evolucionar la app de S5

La app de S5 ya tiene formulario y gateway. Esta demo se centra en lo que ocurre después: reruns, estado persistente, caché del gateway, transiciones y UX de inferencia.

La confianza es una señal del clasificador, no una garantía ni una recomendación profesional.

In [ ]:
from pathlib import Path
import sys

repo_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
                 if (parent / 'semana6/modules/06-ux-model-consumption').is_dir())
solution_src = repo_root / 'semana6/modules/06-ux-model-consumption/solutions/02-robust-streamlit/src'
if str(solution_src) not in sys.path:
    sys.path.insert(0, str(solution_src))

from model_ui.contracts import UiState
from model_ui.controller import PredictionController
from model_ui.gateway import FEATURE_NAMES, DemoGateway
from model_ui.session import clear_last_state, initialize_session_state
from model_ui.telemetry import Telemetry

SAMPLE = {name: 1.0 for name in FEATURE_NAMES}
session_state = {}
initialize_session_state(session_state)
print('claves iniciales:', sorted(session_state))
print('estado inicial:', session_state['last_state'])

## 1. Lo que se pierde en S5

En S5 una variable local como `prediction` solo existe durante una ejecución del script. Si Streamlit vuelve a ejecutar el script porque cambia un widget o se pulsa una acción, esa variable se reconstruye.

Preguntas:

- ¿Qué parte de la app S5 debe sobrevivir?
- ¿Qué parte debe recalcularse?
- ¿Qué objeto debe cargarse una sola vez?
- ¿Qué no debe guardarse en la telemetría?

## 2. Máquina de estados sin Streamlit

El controlador permite observar la secuencia fuera de la UI. El callback `emit` representa el punto donde una app avanzada podría actualizar un placeholder o un `st.status`.

In [ ]:
events = []
telemetry = Telemetry()
controller = PredictionController(
    DemoGateway(confidence=0.74),
    telemetry=telemetry,
)
state = controller.submit(SAMPLE, emit=events.append)
print('transiciones:', [event.phase for event in events])
print('estado final:', state.phase)
print('versiones:', state.view.model_version, state.view.preprocessing_version)
print('telemetría:', telemetry.snapshot())

## 3. Error y recuperación

Un error de contrato no debe destruir el estado de sesión ni mostrar el traceback. La pantalla puede ofrecer reintento y conservar el `request_id` para investigar.

In [ ]:
from model_ui.errors import InputContractError

class BrokenGateway:
    def predict(self, values):
        raise InputContractError('ph fuera de rango')

error_state = PredictionController(
    BrokenGateway(), telemetry=telemetry
).submit(SAMPLE)
print('estado:', error_state.phase)
print('error:', error_state.error.code)
print('recuperación:', error_state.error.recovery)
print('telemetría:', telemetry.snapshot())

## 4. Limpiar sin borrar la observabilidad

Limpiar la última pantalla vuelve a `idle`, pero la telemetría acumulada sigue disponible para la sesión.

In [ ]:
session_state['last_state'] = state
session_state['telemetry'] = telemetry
clear_last_state(session_state)
print('después de limpiar:', session_state['last_state'])
print('peticiones conservadas:', session_state['telemetry'].snapshot().requests_total)

## Preguntas para el debrief

1. ¿Qué diferencia hay entre un recurso cacheado y una respuesta cacheada?
2. ¿Qué claves pondrías en `st.session_state` y cuáles evitarías?
3. ¿Qué parte del controlador podría mantenerse al sustituir el gateway por HTTP en S7?
4. ¿Por qué una latencia alta no convierte necesariamente una predicción válida en un error?